In [ ]:
# Install necessary libraries
!pip install pandas scikit-learn torch transformers sentence-transformers openpyxl spacy

# Download the Spanish spaCy model
!python -m spacy download es_core_news_sm

import os
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import shutil
import torch  # Ensure torch is imported for device handling
import spacy  # Import spaCy for tokenization

# Define the input and output directories
input_directory = '/content/texts_processed'
output_directory = '/content/updated_processed_files'  # New folder to save the updated files

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Load the pre-trained Sentence Transformer model (multilingual)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('distiluse-base-multilingual-cased-v1', device=device)  # Multilingual model

# Load the spaCy Spanish model for tokenization
nlp = spacy.load("es_core_news_sm")

# Function to calculate cosine similarity using CountVectorizer (for POS and Lemmas)
def calculate_cosine_similarity_cv(text1, text2):
    if pd.isna(text1) or pd.isna(text2) or not text1.strip() or not text2.strip():
        return np.nan  # Return NaN if either text is empty or NaN

    # Initialize the vectorizer
    vectorizer = CountVectorizer()

    # Vectorize the texts
    vectors = vectorizer.fit_transform([text1, text2]).toarray()

    # Compute cosine similarity
    try:
        cosine_sim = cosine_similarity([vectors[0]], [vectors[1]])
        return cosine_sim[0, 0]
    except ValueError:
        return np.nan  # Return NaN if there's an issue with the vectors

# Function to calculate cosine similarity using SentenceTransformer embeddings
def calculate_cosine_similarity_embeddings(text1, text2):
    if pd.isna(text1) or pd.isna(text2) or not text1.strip() or not text2.strip():
        return np.nan  # Return NaN if either text is empty or NaN

    # Get embeddings
    embeddings = model.encode([text1, text2], convert_to_tensor=True)

    # Compute cosine similarity
    cosine_sim = cosine_similarity(embeddings[0].cpu().numpy().reshape(1, -1),
                                   embeddings[1].cpu().numpy().reshape(1, -1))
    cosine_sim = np.clip(cosine_sim[0, 0], 0, 1)  # Clip values to ensure they stay within [0, 1]
    return cosine_sim

# Function to calculate lexical density by dividing content words by total words
def lexical_density_from_counts(total_content_words, total_words):
    return total_content_words / total_words if total_words > 0 else 0

# Function to calculate Type-Token Ratio (TTR) for lexical variety
def lexical_variety_from_text(text):
    """
    Calculates the Type-Token Ratio (TTR) for the given text using spaCy tokenization.

    Args:
        text (str): The text to analyze.

    Returns:
        float: The Type-Token Ratio (TTR).
    """
    # Tokenize the text using spaCy
    doc = nlp(text.lower())

    # Extract tokens that are words (alphabetical characters only)
    tokens = [token.text for token in doc if token.is_alpha]

    # Calculate types (unique tokens) and total tokens
    num_tokens = len(tokens)
    num_types = len(set(tokens))

    return num_types / num_tokens if num_tokens > 0 else 0

# Function to calculate length ratio using pre-calculated character counts
def length_ratio(char_count_st, char_count_tr):
    return (char_count_st - char_count_tr) / char_count_st if char_count_st > 0 else 0

# Function to calculate lemma similarity using multilingual embeddings

def calculate_cosine_similarity_multilingual(source_lemmas, trans_lemmas):
    """
    Calculates the cosine similarity between source lemmas and translated lemmas using multilingual embeddings.

    Args:
        source_lemmas (str): Space-separated lemmas from the source text.
        trans_lemmas (str): Space-separated lemmas from the translated text.

    Returns:
        float: Cosine similarity score.
    """
    if not source_lemmas.strip() or not trans_lemmas.strip():
        return np.nan

    # If lemmas are identical, return 1.0
    if source_lemmas.lower() == trans_lemmas.lower():
        return 1.0

    # Tokenize the lemmas
    source_tokens = source_lemmas.lower().split()
    trans_tokens = trans_lemmas.lower().split()

    if not source_tokens or not trans_tokens:
        return np.nan

    # Generate embeddings for the source and translated lemmas
    source_embeddings = model.encode(source_tokens, convert_to_tensor=True)
    trans_embeddings = model.encode(trans_tokens, convert_to_tensor=True)

    # Average the embeddings to obtain a single representation for each set of lemmas
    source_mean = source_embeddings.mean(dim=0).unsqueeze(0).cpu().numpy()
    trans_mean = trans_embeddings.mean(dim=0).unsqueeze(0).cpu().numpy()

    # Calculate cosine similarity between the averaged vectors
    cosine_sim = cosine_similarity(source_mean, trans_mean)[0, 0]

    # Ensure the similarity does not exceed 1.0
    cosine_sim = min(cosine_sim, 1.0)

    return cosine_sim

# Process each Excel file in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith(".xlsx"):
        file_path = os.path.join(input_directory, filename)
        df = pd.read_excel(file_path)

        # Initialize lists to store calculated metrics
        lexical_sims_embeddings_source_trans = []
        lexical_sims_embeddings_mt_trans = []
        lexical_sims_pos_source_trans = []
        lexical_sims_pos_mt_trans = []
        lexical_sims_lemmas_source_trans = []
        lexical_sims_lemmas_mt_trans = []
        lexical_densities = []
        lexical_varieties = []
        length_ratios = []

        # Iterate over each row in the DataFrame
        for index, row in df.iterrows():
            # Get texts
            source_text = str(row.get('SOURCE TEXT', '')).strip()
            target_text = str(row.get('TRANSLATOR RESULT', '')).strip()
            mt_text = str(row.get('MT RESULT', '')).strip()

            # Calculate lexical similarity using embeddings
            lexical_sim_embeddings_source_trans = calculate_cosine_similarity_embeddings(source_text, target_text)
            lexical_sim_embeddings_mt_trans = calculate_cosine_similarity_embeddings(mt_text, target_text)

            # Calculate lexical similarity based on POS
            pos_source = str(row.get('pos_source', '')).strip()
            pos_trans = str(row.get('pos_trans', '')).strip()
            pos_mt = str(row.get('pos_mt', '')).strip()

            lexical_sim_pos_source_trans = calculate_cosine_similarity_cv(pos_source, pos_trans)
            lexical_sim_pos_mt_trans = calculate_cosine_similarity_cv(pos_mt, pos_trans)

            # Calculate lexical similarity using multilingual embeddings
            lemmas_source = str(row.get('lemmas_source', '')).strip()
            lemmas_trans = str(row.get('lemmas_trans', '')).strip()
            lexical_sim_lemmas_source_trans = calculate_cosine_similarity_multilingual(lemmas_source, lemmas_trans)

            lemmas_mt = str(row.get('lemmas_mt', '')).strip()
            lexical_sim_lemmas_mt_trans = calculate_cosine_similarity_multilingual(lemmas_mt, lemmas_trans)

            # Calculate lexical density
            total_content_words = row.get('total content words pos_trans', 0)
            total_words = row.get('WORD COUNT TR', 0)
            lexical_density = lexical_density_from_counts(total_content_words, total_words)

            # Calculate lexical variety using TTR (Type-Token Ratio) on the 'TRANSLATOR RESULT' column
            lexical_variety_val = lexical_variety_from_text(target_text)

            # Calculate length ratio
            char_count_st = row.get('CHAR COUNT ST', 0)
            char_count_tr = row.get('CHAR COUNT TR', 0)
            length_ratio_val = length_ratio(char_count_st, char_count_tr)

            # Append to lists
            lexical_sims_embeddings_source_trans.append(lexical_sim_embeddings_source_trans)
            lexical_sims_embeddings_mt_trans.append(lexical_sim_embeddings_mt_trans)
            lexical_sims_pos_source_trans.append(lexical_sim_pos_source_trans)
            lexical_sims_pos_mt_trans.append(lexical_sim_pos_mt_trans)
            lexical_sims_lemmas_source_trans.append(lexical_sim_lemmas_source_trans)
            lexical_sims_lemmas_mt_trans.append(lexical_sim_lemmas_mt_trans)
            lexical_densities.append(lexical_density)
            lexical_varieties.append(lexical_variety_val)  # Save TTR to Lexical Variety
            length_ratios.append(length_ratio_val)

        # Add new columns to the DataFrame
        df['Lexical Sim Embeddings (source-trans)'] = lexical_sims_embeddings_source_trans
        df['Lexical Sim Embeddings (mt-trans)'] = lexical_sims_embeddings_mt_trans
        df['Lexical Sim POS (source-trans)'] = lexical_sims_pos_source_trans
        df['Lexical Sim POS (mt-trans)'] = lexical_sims_pos_mt_trans
        df['Lexical Sim Lemmas (source-trans)'] = lexical_sims_lemmas_source_trans
        df['Lexical Sim Lemmas (mt-trans)'] = lexical_sims_lemmas_mt_trans
        df['Lexical Density'] = lexical_densities
        df['Lexical Variety'] = lexical_varieties  # Add TTR to the 'Lexical Variety' column
        df['Length Ratio'] = length_ratios

        # Save the updated DataFrame to the output directory
        output_file_path = os.path.join(output_directory, 'updated_' + filename)
        df.to_excel(output_file_path, index=False)

        print(f"Metrics added and saved to {output_file_path}")

print("All files processed.")

# Step to create a zip file containing all processed files
output_zip_path = '/content/updated_processed_texts.zip'
shutil.make_archive(output_zip_path.replace('.zip', ''), 'zip', output_directory)

print(f"All processed files have been zipped into: {output_zip_path}")

